In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import f1_score


In [2]:
val_binary = pd.read_csv("Extraction/val_binary.tsv", sep="\t")
test_binary = pd.read_csv("Extraction/test_binary.tsv", sep="\t")
metadata_cols = ["Entry", "Length", "Sequence"]
concept_cols = [c for c in val_binary.columns if c not in metadata_cols]

In [5]:
def compare_features_loop(A, binary_df, concept_cols, thresholds=(0, 0.15, 0.5, 0.6, 0.8)):
    Y = binary_df[concept_cols].values.astype(int)
    results = []
    for c_idx, concept in enumerate(concept_cols):
        y = Y[:, c_idx]
        for f in range(A.shape[1]):
            for t in thresholds:
                pred = (A[:, f] > t).astype(int)
                tp = ((pred == 1) & (y == 1)).sum()
                fp = ((pred == 1) & (y == 0)).sum()
                if tp == 0:
                    continue
                precision = tp / (tp + fp)
                recall = tp / y.sum() if y.sum() > 0 else 0
                f1 = f1_score(y, pred, zero_division=0)
                results.append({
                    "concept": concept,
                    "feature": f,
                    "threshold": t,
                    "precision": precision,
                    "recall": recall,
                    "f1": f1,
                    #"tp": tp,
                    #"fp": fp,
                    #"positive_labels": y.sum(),
                })
    return pd.DataFrame(results)

In [ ]:
def calculate_f1_array(precision, recall):
    denom = precision + recall
    return np.divide(
        2 * precision * recall,
        denom,
        out=np.zeros_like(denom, dtype=float),
        where=denom > 0
    )

def compare_features_to_concepts_fast(
    A,
    binary_df,
    concept_cols,
    thresholds=(0, 0.15, 0.5, 0.6, 0.8),
):
    """
    A: normalized activations, shape [n_proteins, n_features]
    binary_df: val_binary/test_binary
    concept_cols: concept label columns

    Returns dataframe with:
    concept, feature, threshold, precision, recall, f1
    """

    A = np.asarray(A)
    Y = binary_df[concept_cols].values.astype(bool)

    n_proteins, n_features = A.shape
    n_concepts = Y.shape[1]

    positives = Y.sum(axis=0)  # [n_concepts]
    results = []

    for threshold in thresholds:
        A_bin = A > threshold  # [n_proteins, n_features]

        # Matrix multiplication gives TP:
        # Y.T: [n_concepts, n_proteins]
        # A_bin: [n_proteins, n_features]
        # tp: [n_concepts, n_features]
        tp = Y.T.astype(np.int32) @ A_bin.astype(np.int32)

        pred_pos = A_bin.sum(axis=0)  # [n_features]
        fp = pred_pos[None, :] - tp

        precision = np.divide(
            tp,
            tp + fp,
            out=np.zeros_like(tp, dtype=float),
            where=(tp + fp) > 0,
        )

        recall = np.divide(
            tp,
            positives[:, None],
            out=np.zeros_like(tp, dtype=float),
            where=positives[:, None] > 0,
        )

        f1 = calculate_f1_array(precision, recall)

        concept_idx, feature_idx = np.nonzero(tp > 0)

        df_t = pd.DataFrame({
            "concept": [concept_cols[i] for i in concept_idx],
            "feature": feature_idx,
            "threshold": threshold,
            "precision": precision[concept_idx, feature_idx],
            "recall": recall[concept_idx, feature_idx],
            "f1": f1[concept_idx, feature_idx],
            #"tp": tp[concept_idx, feature_idx],
            #"fp": fp[concept_idx, feature_idx],
            #"positive_labels": positives[concept_idx],
        })

        results.append(df_t)

    return pd.concat(results, ignore_index=True)

CLS 8

In [15]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_8/embeddings_cls_ica_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_8/embeddings_cls_ica_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

(25000, 8)
(25000, 8)


In [16]:
val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})

In [19]:
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))


val f1: 0.06183
test f1: 0.05819


In [20]:
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)

Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


In [ ]:
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)


CLS 32

In [22]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_32/embeddings_cls_ica_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_32/embeddings_cls_ica_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.10378
test f1: 0.09712
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 2
Survival rate: 0.5


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
2,GO:0005634,23,0.5,0.538367,0.534049,0.536199,0.540199,0.539758,0.539978
0,GO:0003924,11,0.5,0.678922,0.505474,0.579498,0.628415,0.438931,0.516854


ica 128

In [23]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_128/embeddings_cls_ica_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_128/embeddings_cls_ica_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.13057
test f1: 0.1202
Validation pairs with F1 > 0.5: 8
Those also with test F1 > 0.5: 6
Survival rate: 0.75


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,127,0.5,0.833703,0.686131,0.752753,0.832524,0.654580,0.732906
2,GO:0005525,127,0.5,0.900222,0.539894,0.674979,0.898058,0.527817,0.664870
1,GO:0003925,127,0.6,0.548295,0.881279,0.676007,0.486928,0.818681,0.610656
3,GO:0005506,96,0.5,0.776557,0.515815,0.619883,0.756198,0.491935,0.596091
4,heme,96,0.5,0.754579,0.490476,0.594517,0.727273,0.458333,0.562300
5,1.14,96,0.5,0.641026,0.432099,0.516224,0.640496,0.411141,0.500808


In [44]:
val_data = np.load(
    "Extraction/val_features/embeddings_cls_320/embeddings_cls_ica_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_cls_320/embeddings_cls_ica_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.13222
test f1: 0.1209
Validation pairs with F1 > 0.5: 6
Those also with test F1 > 0.5: 5
Survival rate: 0.8333333333333334


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,krab,182,0.8,0.677966,0.769231,0.720721,0.711864,0.688525,0.700000
2,ig-like,241,0.6,0.675556,0.716981,0.695652,0.676399,0.676399,0.676399
1,GO:0003925,250,0.6,0.648649,0.767123,0.702929,0.577093,0.719780,0.640587
3,pyridoxal 5'-phosphate,13,0.5,0.887850,0.416667,0.567164,0.961538,0.421941,0.586510
4,GO:0003924,250,0.6,0.864865,0.408759,0.555143,0.845815,0.366412,0.511318


Layer Mean

In [24]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_8/embeddings_layer_mean_ica_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_8/embeddings_layer_mean_ica_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.07385
test f1: 0.07033
Validation pairs with F1 > 0.5: 1
Those also with test F1 > 0.5: 1
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,transmembrane,6,0.15,0.383558,0.895912,0.537151,0.384949,0.896983,0.538707


In [25]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_16/embeddings_layer_mean_ica_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_16/embeddings_layer_mean_ica_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.09278
test f1: 0.08821
Validation pairs with F1 > 0.5: 3
Those also with test F1 > 0.5: 2
Survival rate: 0.6666666666666666


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,14,0.8,0.350142,0.90227,0.504503,0.352686,0.907435,0.507951
1,GO:0007186,3,0.6,0.542936,0.46890,0.503209,0.548476,0.469194,0.505747


In [26]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_32/embeddings_layer_mean_ica_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_32/embeddings_layer_mean_ica_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.12682
test f1: 0.12299
Validation pairs with F1 > 0.5: 13
Those also with test F1 > 0.5: 12
Survival rate: 0.9230769230769231


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,abc transporter,2,0.50,0.918367,0.803571,0.857143,0.895522,0.740741,0.810811
1,GO:0003924,26,0.50,0.681574,0.695255,0.688347,0.708696,0.622137,0.662602
2,GO:0005506,5,0.60,0.976636,0.508516,0.668800,0.939394,0.500000,0.652632
5,GO:0005525,26,0.50,0.742397,0.551862,0.633105,0.784783,0.514979,0.621878
4,heme,5,0.60,0.962617,0.490476,0.649842,0.909091,0.468750,0.618557
3,GO:0003925,26,0.60,0.530055,0.885845,0.663248,0.472050,0.835165,0.603175
6,transmembrane,7,0.15,0.507453,0.654930,0.571836,0.508284,0.652040,0.571257
7,GO:0005634,24,0.50,0.485584,0.652440,0.556780,0.485711,0.642245,0.553117
8,1.14,5,0.60,0.803738,0.424691,0.555735,0.782828,0.411141,0.539130
9,GO:0020037,5,0.60,0.976636,0.384191,0.551451,0.939394,0.359073,0.519553


In [27]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_64/embeddings_layer_mean_ica_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_64/embeddings_layer_mean_ica_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.12733
test f1: 0.1204
Validation pairs with F1 > 0.5: 13
Those also with test F1 > 0.5: 11
Survival rate: 0.8461538461538461


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,8,0.6,0.785173,0.922772,0.848430,0.793589,0.926941,0.855097
1,abc transporter,38,0.5,1.000000,0.696429,0.821053,0.963636,0.654321,0.779412
3,GO:0106310,8,0.6,0.617523,0.911692,0.736313,0.613761,0.924617,0.737782
2,ig-like,30,0.6,0.850144,0.695755,0.765240,0.835404,0.654501,0.733970
5,GO:0004674,8,0.6,0.543387,0.865772,0.667702,0.560594,0.894015,0.689092
4,GO:0005506,14,0.5,1.000000,0.508516,0.674194,0.994652,0.500000,0.665474
7,2.7,8,0.6,0.761584,0.554261,0.641590,0.758405,0.568915,0.650134
6,heme,14,0.5,0.985646,0.490476,0.655008,0.962567,0.468750,0.630473
8,n-acetyltransferase,9,0.8,0.909091,0.461538,0.612245,0.896552,0.448276,0.597701
9,1.14,14,0.5,0.822967,0.424691,0.560261,0.828877,0.411141,0.549645


In [28]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_128/embeddings_layer_mean_ica_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_128/embeddings_layer_mean_ica_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.15835
test f1: 0.14824
Validation pairs with F1 > 0.5: 20
Those also with test F1 > 0.5: 18
Survival rate: 0.9


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,92,0.5,0.987879,0.968317,0.978000,0.982226,0.958904,0.970425
1,GO:0106310,92,0.5,0.777778,0.957711,0.858417,0.748363,0.942285,0.834202
4,fad-binding fr-type,80,0.6,0.681159,0.903846,0.776860,0.690141,0.924528,0.790323
3,GO:0004674,92,0.5,0.685859,0.911409,0.782709,0.681010,0.907731,0.778193
2,GO:0003924,87,0.5,0.868534,0.735401,0.796443,0.876263,0.662214,0.754348
6,2.7,92,0.5,0.946465,0.574494,0.714994,0.929841,0.582991,0.716655
5,GO:0003925,87,0.8,0.697417,0.863014,0.771429,0.629787,0.813187,0.709832
7,GO:0005525,87,0.5,0.933190,0.575798,0.712171,0.949495,0.536377,0.685506
8,GO:0005506,50,0.5,1.000000,0.508516,0.674194,0.994652,0.500000,0.665474
9,heme,50,0.5,0.985646,0.490476,0.655008,0.962567,0.468750,0.630473


In [45]:
val_data = np.load(
    "Extraction/val_features/embeddings_layer_mean_320/embeddings_layer_mean_ica_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_layer_mean_320/embeddings_layer_mean_ica_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.17493
test f1: 0.16729
Validation pairs with F1 > 0.5: 23
Those also with test F1 > 0.5: 21
Survival rate: 0.9130434782608695


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
1,c-type lectin,26,0.50,1.000000,0.767442,0.868421,1.000000,0.855422,0.922078
2,fad-binding fr-type,21,0.50,1.000000,0.730769,0.844444,1.000000,0.773585,0.872340
0,6.2,39,0.60,0.915493,0.833333,0.872483,0.898305,0.779412,0.834646
5,rrm,291,0.50,0.975000,0.626506,0.762836,0.982558,0.647510,0.780600
3,protein kinase,230,0.50,1.000000,0.633663,0.775758,0.995726,0.638356,0.777963
4,GO:0003924,303,0.50,0.875576,0.693431,0.773931,0.886598,0.656489,0.754386
7,GO:0106310,230,0.50,0.821875,0.654229,0.728532,0.810541,0.670200,0.733720
8,pyridoxal 5'-phosphate,182,0.50,1.000000,0.552632,0.711864,1.000000,0.578059,0.732620
6,GO:0003925,246,0.60,0.684982,0.853881,0.760163,0.634454,0.829670,0.719048
15,krab,56,0.60,0.596774,0.711538,0.649123,0.728814,0.704918,0.716667


Max

In [29]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_8/embeddings_max_ica_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_8/embeddings_max_ica_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.07107
test f1: 0.06866
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [30]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_16/embeddings_max_ica_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_16/embeddings_max_ica_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.08491
test f1: 0.08155
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,abc transporter,9,0.8,0.965517,0.500000,0.658824,1.000000,0.370370,0.540541
1,GO:0005634,11,0.6,0.338238,0.968601,0.501390,0.339565,0.970232,0.503066


In [31]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_32/embeddings_max_ica_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_32/embeddings_max_ica_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.10725
test f1: 0.10179
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 3
Survival rate: 0.75


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,abc transporter,4,0.6,0.982456,1.000000,0.991150,0.941176,0.987654,0.963855
1,pyridoxal 5'-phosphate,21,0.8,0.991228,0.495614,0.660819,0.983471,0.502110,0.664804
3,GO:0007186,5,0.8,0.535248,0.490431,0.511860,0.540761,0.471564,0.503797


In [32]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_64/embeddings_max_ica_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_64/embeddings_max_ica_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.15055
test f1: 0.14446
Validation pairs with F1 > 0.5: 15
Those also with test F1 > 0.5: 15
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,56,0.5,0.888889,0.861314,0.874884,0.896761,0.845420,0.870334
2,GO:0005525,56,0.5,0.969868,0.684840,0.802806,0.975709,0.687589,0.806695
1,pyridoxal 5'-phosphate,62,0.6,0.993590,0.679825,0.807292,0.964072,0.679325,0.797030
5,j,31,0.8,0.891892,0.600000,0.717391,0.860000,0.716667,0.781818
3,ig-like,37,0.6,0.712551,0.830189,0.766885,0.676349,0.793187,0.730123
4,rrm,21,0.6,0.626374,0.915663,0.743883,0.592593,0.919540,0.720721
6,fad,7,0.6,0.925581,0.560563,0.698246,0.881057,0.560224,0.684932
7,GO:0005506,26,0.8,1.000000,0.484185,0.652459,1.000000,0.467742,0.637363
8,heme,26,0.8,0.984925,0.466667,0.633279,0.965517,0.437500,0.602151
10,GO:0004930,2,0.5,0.427252,0.872642,0.573643,0.395294,0.888889,0.547231


In [33]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_128/embeddings_max_ica_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_128/embeddings_max_ica_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.1528
test f1: 0.14535
Validation pairs with F1 > 0.5: 14
Those also with test F1 > 0.5: 14
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,rrm,85,0.6,0.978070,0.895582,0.935010,0.991667,0.911877,0.950100
1,helicase,52,0.5,0.974359,0.844444,0.904762,0.924528,0.864706,0.893617
2,ig-like,80,0.6,0.933148,0.790094,0.855683,0.921922,0.746959,0.825269
4,pyridoxal 5'-phosphate,58,0.5,0.957317,0.688596,0.801020,0.941520,0.679325,0.789216
5,n-acetyltransferase,15,0.8,0.882353,0.692308,0.775862,0.904762,0.655172,0.760000
6,ef-hand,113,0.5,0.970588,0.630573,0.764479,0.960000,0.615385,0.750000
7,GO:0061630,71,0.5,0.745413,0.719027,0.731982,0.755784,0.727723,0.741488
8,krab,40,0.8,0.630769,0.788462,0.700855,0.684211,0.639344,0.661017
3,kh,19,0.5,0.802469,0.822785,0.812500,0.734694,0.590164,0.654545
10,2.1,78,0.5,0.676991,0.533101,0.596491,0.744292,0.541528,0.626923


In [46]:
val_data = np.load(
    "Extraction/val_features/embeddings_max_320/embeddings_max_ica_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_max_320/embeddings_max_ica_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.15329
test f1: 0.14267
Validation pairs with F1 > 0.5: 15
Those also with test F1 > 0.5: 13
Survival rate: 0.8666666666666667


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,abc transporter,109,0.5,1.000000,1.000000,1.000000,1.000000,1.000000,1.000000
1,fad-binding fr-type,42,0.6,1.000000,0.961538,0.980392,1.000000,0.962264,0.980769
3,helicase,24,0.5,0.986577,0.816667,0.893617,0.966887,0.858824,0.909657
2,j,135,0.5,1.000000,0.818182,0.900000,1.000000,0.816667,0.899083
4,GO:0061630,110,0.5,0.846797,0.672566,0.749692,0.857567,0.715347,0.780027
5,lrrct,133,0.6,0.619718,0.871287,0.724280,0.594595,0.846154,0.698413
6,GO:0004252,33,0.5,0.959538,0.562712,0.709402,0.964497,0.532680,0.686316
7,lrrnt,133,0.6,0.464789,0.891892,0.611111,0.425676,0.840000,0.565022
11,peptidase,33,0.5,0.971098,0.400000,0.566610,0.958580,0.384798,0.549153
9,ubiquitin-like,32,0.6,0.678571,0.506667,0.580153,0.659091,0.453125,0.537037


Mean

In [34]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_8/embeddings_mean_ica_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_8/embeddings_mean_ica_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.07126
test f1: 0.06918
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0005634,7,0.5,0.378022,0.965067,0.543250,0.378653,0.961533,0.543339
1,GO:0005737,7,0.5,0.355447,0.909041,0.511062,0.363612,0.914266,0.520297


In [35]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_16/embeddings_mean_ica_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_16/embeddings_mean_ica_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.07999
test f1: 0.07629
Validation pairs with F1 > 0.5: 2
Those also with test F1 > 0.5: 2
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,ig-like,9,0.8,0.612308,0.469340,0.531375,0.607362,0.481752,0.537313
1,GO:0005634,11,0.8,0.433495,0.642789,0.517793,0.444740,0.658556,0.530930


In [36]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_32/embeddings_mean_ica_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_32/embeddings_mean_ica_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.11351
test f1: 0.10827
Validation pairs with F1 > 0.5: 3
Those also with test F1 > 0.5: 3
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003925,3,0.5,0.533898,0.863014,0.659686,0.488673,0.829670,0.615071
2,pyridoxal 5'-phosphate,7,0.8,0.624204,0.429825,0.509091,0.769231,0.421941,0.544959
1,GO:0003924,3,0.5,0.703390,0.454380,0.552106,0.695793,0.410305,0.516206


In [37]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_64/embeddings_mean_ica_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_64/embeddings_mean_ica_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.14055
test f1: 0.13422
Validation pairs with F1 > 0.5: 14
Those also with test F1 > 0.5: 12
Survival rate: 0.8571428571428571


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,pyridoxal 5'-phosphate,22,0.60,0.986667,0.649123,0.783069,0.969325,0.666667,0.790000
1,GO:0003925,33,0.80,0.673759,0.867580,0.758483,0.599206,0.829670,0.695853
2,GO:0003924,33,0.60,0.687279,0.709854,0.698384,0.679208,0.654580,0.666667
4,GO:0005506,12,0.80,1.000000,0.506083,0.672052,1.000000,0.494624,0.661871
3,ig-like,48,0.60,0.622857,0.771226,0.689146,0.593023,0.744526,0.660194
6,GO:0005525,33,0.60,0.750883,0.565160,0.644917,0.748515,0.539230,0.626866
5,heme,12,0.80,0.985577,0.488095,0.652866,0.967391,0.463542,0.626761
10,2.6,22,0.80,0.530864,0.500000,0.514970,0.581395,0.568182,0.574713
7,1.14,12,0.80,0.822115,0.422222,0.557912,0.842391,0.411141,0.552585
9,transmembrane,37,0.15,0.419976,0.712126,0.528355,0.418488,0.702263,0.524450


In [38]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_128/embeddings_mean_ica_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_128/embeddings_mean_ica_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.16512
test f1: 0.15631
Validation pairs with F1 > 0.5: 18
Those also with test F1 > 0.5: 15
Survival rate: 0.8333333333333334


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,34,0.6,0.955603,0.895050,0.924335,0.939365,0.891324,0.914714
1,helicase,85,0.6,0.884393,0.850000,0.866856,0.827778,0.876471,0.851429
2,GO:0106310,34,0.6,0.758985,0.893035,0.820571,0.716073,0.876325,0.788136
3,ig-like,13,0.6,0.877193,0.707547,0.783290,0.859425,0.654501,0.743094
4,GO:0004674,34,0.6,0.664905,0.844295,0.743938,0.645813,0.836658,0.728952
6,2.7,34,0.6,0.928118,0.538320,0.681412,0.912416,0.556012,0.690962
5,fad,125,0.5,0.933962,0.557746,0.698413,0.908257,0.554622,0.688696
9,abc transporter,109,0.5,1.000000,0.500000,0.666667,0.977273,0.530864,0.688000
7,GO:0005506,44,0.5,1.000000,0.508516,0.674194,0.994652,0.500000,0.665474
10,heme,44,0.5,0.985646,0.490476,0.655008,0.962567,0.468750,0.630473


In [47]:
val_data = np.load(
    "Extraction/val_features/embeddings_mean_320/embeddings_mean_ica_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_mean_320/embeddings_mean_ica_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.15365
test f1: 0.14235
Validation pairs with F1 > 0.5: 14
Those also with test F1 > 0.5: 13
Survival rate: 0.9285714285714286


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,helicase,209,0.5,0.950617,0.855556,0.900585,0.943396,0.882353,0.911854
2,fad-binding fr-type,187,0.5,1.000000,0.711538,0.831461,1.000000,0.792453,0.884211
4,c-type lectin,299,0.5,1.000000,0.674419,0.805556,1.000000,0.746988,0.855172
1,6.2,67,0.5,0.915493,0.833333,0.872483,0.883333,0.779412,0.828125
3,n-acetyltransferase,117,0.5,0.924528,0.753846,0.830508,0.972973,0.620690,0.757895
5,GO:0003925,184,0.5,0.675862,0.894977,0.770138,0.601562,0.846154,0.703196
8,lrrct,155,0.5,0.511111,0.683168,0.584746,0.506494,0.750000,0.604651
6,GO:0003924,184,0.5,0.917241,0.485401,0.634845,0.917969,0.448473,0.602564
7,GO:0004252,0,0.5,0.992424,0.444068,0.613583,0.992424,0.428105,0.598174
10,GO:0005525,184,0.5,1.000000,0.385638,0.556622,1.000000,0.365193,0.535005


Min

In [39]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_8/embeddings_min_ica_8_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_8/embeddings_min_ica_8_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 8)
(25000, 8)
val f1: 0.08565
test f1: 0.08316
Validation pairs with F1 > 0.5: 4
Those also with test F1 > 0.5: 4
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,GO:0003924,0,0.8,0.676587,0.622263,0.648289,0.704805,0.587786,0.640999
1,GO:0005525,0,0.8,0.751984,0.503989,0.603503,0.780320,0.486448,0.599297
2,protein kinase,3,0.8,0.456356,0.771287,0.573427,0.471730,0.754338,0.580464
3,GO:0005634,5,0.6,0.442367,0.678673,0.535615,0.446661,0.681800,0.539732


In [40]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_16/embeddings_min_ica_16_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_16/embeddings_min_ica_16_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 16)
(25000, 16)
val f1: 0.08568
test f1: 0.08337
Validation pairs with F1 > 0.5: 0
Those also with test F1 > 0.5: 0
Survival rate: 0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1


In [41]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_32/embeddings_min_ica_32_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_32/embeddings_min_ica_32_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 32)
(25000, 32)
val f1: 0.12674
test f1: 0.12174
Validation pairs with F1 > 0.5: 9
Those also with test F1 > 0.5: 9
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,18,0.8,0.539140,0.968317,0.692635,0.561804,0.967123,0.710738
1,Zinc finger,24,0.5,0.767205,0.568998,0.653401,0.764398,0.556544,0.644118
3,GO:0106310,18,0.8,0.423925,0.956468,0.587471,0.435544,0.967020,0.600585
2,GO:0016887,8,0.6,0.625632,0.562121,0.592179,0.623064,0.549317,0.583871
4,pyridoxal 5'-phosphate,9,0.8,0.703947,0.469298,0.563158,0.773050,0.459916,0.576720
6,2.7,18,0.8,0.522602,0.581239,0.550363,0.538992,0.595894,0.566017
7,GO:0004674,18,0.8,0.377067,0.918121,0.534584,0.398408,0.936409,0.558988
5,GO:0008270,24,0.5,0.802039,0.424460,0.555131,0.788831,0.401599,0.532234
8,GO:0005634,26,0.5,0.511939,0.504146,0.508013,0.508889,0.498029,0.503400


In [42]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_64/embeddings_min_ica_64_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_64/embeddings_min_ica_64_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 64)
(25000, 64)
val f1: 0.14852
test f1: 0.13894
Validation pairs with F1 > 0.5: 15
Those also with test F1 > 0.5: 15
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,abc transporter,47,0.6,1.000000,1.000000,1.000000,0.987805,1.000000,0.993865
1,protein kinase,31,0.8,0.745606,0.798020,0.770923,0.751661,0.826484,0.787299
2,helicase,9,0.8,1.000000,0.611111,0.758621,0.970874,0.588235,0.732601
4,GO:0106310,31,0.8,0.607771,0.817164,0.697082,0.597176,0.846879,0.700438
3,ig-like,55,0.6,0.652174,0.813679,0.724029,0.610390,0.800487,0.692632
5,fad,53,0.6,0.929245,0.554930,0.694885,0.896861,0.560224,0.689655
6,Zinc finger,36,0.5,0.638764,0.716446,0.675379,0.629484,0.713469,0.668851
7,GO:0004674,31,0.8,0.547641,0.794631,0.648412,0.545681,0.819202,0.655035
10,2.1,51,0.5,0.643443,0.547038,0.591337,0.732143,0.544850,0.624762
8,nudix hydrolase,19,0.8,0.480392,0.890909,0.624204,0.459184,0.882353,0.604027


In [43]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_128/embeddings_min_ica_128_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_128/embeddings_min_ica_128_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 128)
(25000, 128)
val f1: 0.15536
test f1: 0.14714
Validation pairs with F1 > 0.5: 17
Those also with test F1 > 0.5: 17
Survival rate: 1.0


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
0,protein kinase,25,0.5,0.993076,0.994059,0.993568,0.992687,0.991781,0.992234
1,GO:0106310,25,0.6,0.789528,0.956468,0.865017,0.775895,0.970554,0.862376
4,f-box,122,0.6,0.859649,0.720588,0.784000,0.875000,0.753846,0.809917
3,GO:0004674,25,0.6,0.697125,0.911409,0.789994,0.706215,0.935162,0.804721
5,GO:0061630,119,0.6,0.837838,0.685841,0.754258,0.861027,0.705446,0.775510
2,ef-hand,115,0.5,0.936170,0.700637,0.801457,0.904040,0.655678,0.760085
6,2.7,25,0.5,0.944609,0.585530,0.722937,0.934186,0.599413,0.730261
8,GO:0005506,79,0.6,1.000000,0.508516,0.674194,1.000000,0.500000,0.666667
7,GO:0003925,75,0.6,0.614887,0.867580,0.719697,0.553114,0.829670,0.663736
9,heme,79,0.6,0.985646,0.490476,0.655008,0.967742,0.468750,0.631579


In [48]:
val_data = np.load(
    "Extraction/val_features/embeddings_min_320/embeddings_min_ica_320_val_features.npz",
    allow_pickle=True,
)

test_data = np.load(
    "Extraction/test_features/embeddings_min_320/embeddings_min_ica_320_test_features.npz",
    allow_pickle=True,
)
X_val = val_data["features"]
X_test = test_data["features"]
print(X_val.shape)
print(X_test.shape)

mins = X_val.min(axis=0)
maxs = X_val.max(axis=0)

A_val = (X_val - mins) / (maxs - mins + 1e-8)
A_test = (X_test - mins) / (maxs - mins + 1e-8)

val_metrics = compare_features_to_concepts_fast(
    A_val,
    val_binary,
    concept_cols
)
best_val = (
    val_metrics
    .sort_values("f1", ascending=False)
    .groupby("concept")
    .head(1)
    .reset_index(drop=True)
)
best_val = best_val.rename(columns={
    "precision": "val_precision",
    "recall": "val_recall",
    "f1": "val_f1",
})
test_metrics = compare_features_to_concepts_fast(
    A_test,
    test_binary,
    concept_cols
)
selected_test = best_val[["concept", "feature", "threshold", "val_precision", "val_recall", "val_f1"]].merge(
    test_metrics,
    on=["concept", "feature", "threshold"],
    how="left"
)
selected_test = selected_test.rename(columns={
    "precision": "test_precision",
    "recall": "test_recall",
    "f1": "test_f1",
})
print("val f1:", round(selected_test["val_f1"].mean(),5))
print("test f1:", round(selected_test["test_f1"].fillna(0).mean(),5))
num_val_pairs = (selected_test["val_f1"] > 0.5).sum()
num_test_survived = (
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
).sum()
print("Validation pairs with F1 > 0.5:", num_val_pairs)
print("Those also with test F1 > 0.5:", num_test_survived)
print("Survival rate:", num_test_survived / num_val_pairs if num_val_pairs > 0 else 0)
surviving_pairs = selected_test[
    (selected_test["val_f1"] > 0.5) &
    (selected_test["test_f1"].fillna(0) > 0.5)
].sort_values("test_f1", ascending=False)
surviving_pairs


(25000, 320)
(25000, 320)
val f1: 0.20823
test f1: 0.19523
Validation pairs with F1 > 0.5: 29
Those also with test F1 > 0.5: 27
Survival rate: 0.9310344827586207


,concept,feature,threshold,val_precision,val_recall,val_f1,test_precision,test_recall,test_f1
2,nudix hydrolase,126,0.5,1.000000,0.927273,0.962264,1.000000,1.000000,1.000000
1,c-type lectin,256,0.5,1.000000,0.965116,0.982249,1.000000,0.987952,0.993939
0,protein kinase,225,0.5,0.995992,0.984158,0.990040,0.996293,0.981735,0.988960
3,rrm,317,0.5,0.929412,0.951807,0.940476,0.932584,0.954023,0.943182
6,j,37,0.5,1.000000,0.800000,0.888889,1.000000,0.866667,0.928571
4,cadherin,36,0.5,0.880597,1.000000,0.936508,0.879310,0.980769,0.927273
5,helicase,319,0.5,1.000000,0.816667,0.899083,0.972222,0.823529,0.891720
9,GO:0106310,225,0.5,0.779559,0.967662,0.863485,0.766450,0.974087,0.857884
7,6.2,276,0.6,0.928571,0.833333,0.878378,0.898305,0.779412,0.834646
12,GO:0004674,225,0.6,0.699170,0.904698,0.788765,0.706553,0.927681,0.802156
